# Kaggle: Predicting Loan Payback

Load libraries used in notebook

In [17]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

### 1. Data

Read the datasets into pandas DataFrames.

In [2]:
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

Display the DataFrames.

In [3]:
display(train_df)
display(test_df)

,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back
0,0,29367.99,0.084,736,2528.42,13.67,Female,Single,High School,Self-employed,Other,C3,1.0
1,1,22108.02,0.166,636,4593.10,12.92,Male,Married,Master's,Employed,Debt consolidation,D3,0.0
2,2,49566.20,0.097,694,17005.15,9.76,Male,Single,High School,Employed,Debt consolidation,C5,1.0
3,3,46858.25,0.065,533,4682.48,16.10,Female,Single,High School,Employed,Debt consolidation,F1,1.0
4,4,25496.70,0.053,665,12184.43,10.21,Male,Married,High School,Employed,Other,D1,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
593989,593989,23004.26,0.152,703,20958.37,10.92,Female,Single,High School,Employed,Business,C3,1.0
593990,593990,35289.43,0.105,559,3257.24,14.62,Male,Single,Bachelor's,Employed,Debt consolidation,F5,1.0
593991,593991,47112.64,0.072,675,929.27,14.13,Female,Married,Bachelor's,Employed,Debt consolidation,C1,1.0
593992,593992,76748.44,0.067,740,16290.40,9.87,Male,Single,Bachelor's,Employed,Debt consolidation,B2,1.0


,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade
0,593994,28781.05,0.049,626,11461.42,14.73,Female,Single,High School,Employed,Other,D5
1,593995,46626.39,0.093,732,15492.25,12.85,Female,Married,Master's,Employed,Other,C1
2,593996,54954.89,0.367,611,3796.41,13.29,Male,Single,Bachelor's,Employed,Debt consolidation,D1
3,593997,25644.63,0.110,671,6574.30,9.57,Female,Single,Bachelor's,Employed,Debt consolidation,C3
4,593998,25169.64,0.081,688,17696.89,12.80,Female,Married,PhD,Employed,Business,C1
...,...,...,...,...,...,...,...,...,...,...,...,...
254564,848558,92835.97,0.068,744,29704.00,13.48,Female,Single,Bachelor's,Employed,Debt consolidation,B2
254565,848559,48846.47,0.091,634,20284.33,9.58,Female,Married,High School,Employed,Debt consolidation,D4
254566,848560,20668.52,0.096,718,26387.55,9.00,Male,Single,Master's,Employed,Debt consolidation,C4
254567,848561,34105.09,0.094,739,11107.36,9.81,Male,Single,Bachelor's,Employed,Business,C2


Display categorical features.

In [4]:
text_columns = train_df.select_dtypes(exclude='number').columns
print("Potential categorical columns:", text_columns.tolist())

Potential categorical columns: ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']


## 2. Feature Matrix and Target Vector Definition

Build feature matrix and label vector.

In [5]:
X = train_df.select_dtypes(include='number').drop(columns=['id', 'loan_paid_back'])
y = train_df['loan_paid_back']

Print the shapes and features names.

In [6]:
print(X.shape, y.shape)
print(X.columns.tolist())

(593994, 5) (593994,)
['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']


## 3. Train/Test Data Split

Split the data into testing and training sets.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.20, random_state=9)

## 4. Baseline Models

### Logistic Regression

Create Logistic Regression Classifier.

In [8]:
my_logreg = LogisticRegression(max_iter=1000)
my_logreg.fit(X_train,y_train)

probability = my_logreg.predict_proba(X_test)[:,1]
roc_auc_score(y_test,probability)

0.7735132575141423

### KNN Classifier

Create KNN Classifiers.

In [9]:
scaler = StandardScaler()
X_train_normalized = scaler.fit_transform(X_train)
X_test_normalized = scaler.transform(X_test)

k_values = [3, 5, 11, 15, 25 ,30 ,35, 45, 55, 75, 95, 125, 155, 200, 255]
auc_scores = []

for k in k_values:
    my_knn = KNeighborsClassifier(n_neighbors=k)
    my_knn.fit(X_train_normalized, y_train)
    probability = my_knn.predict_proba(X_test_normalized)[:, 1]
    auc = roc_auc_score(y_test, probability)
    auc_scores.append(auc)
    
for k, score in zip(k_values, auc_scores):
    print(f"K={k}: AUC {score:.4f}")

K=3: AUC 0.6609
K=5: AUC 0.6939
K=11: AUC 0.7293
K=15: AUC 0.7415
K=25: AUC 0.7565
K=30: AUC 0.7606
K=35: AUC 0.7631
K=45: AUC 0.7667
K=55: AUC 0.7691
K=75: AUC 0.7717
K=95: AUC 0.7734
K=125: AUC 0.7750
K=155: AUC 0.7757
K=200: AUC 0.7765
K=255: AUC 0.7770


### Decision Tree Classifier

Create Decision Tree Classifier.

In [15]:
dt = DecisionTreeClassifier(random_state=5)

dt.fit(X_train, y_train)

probability_dt = dt.predict_proba(X_test)[:, 1]
auc_dt = roc_auc_score(y_test, probability_dt)
print("Decision Tree AUC:", auc_dt)

Decision Tree AUC: 0.7740693730322125


### Random Forest Classifier

In [ ]:
tree_counts = [100, 200, 300, 500]
rf_auc_scores = []

for n in tree_counts:
    my_rf = RandomForestClassifier(n_estimators=n, class_weight='balanced', random_state=5)
    
    my_rf.fit(X_train, y_train)
    
    probability = my_rf.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, probability)
    rf_auc_scores.append(auc)

for n, score in zip(tree_counts, rf_auc_scores):
    print(f"n_estimators={n}: AUC {score:.4f}")

In [ ]:
print("Final Model AUC Scores:")
print("Baseline Logistic Regression AUC: 0.7735")
print("Encoded Logistic Regression AUC:", auc_enc)
print("KNN AUC:", max(auc_scores))
print("Decision Tree AUC:", auc_dt)
print("Random Forest AUC:", auc_rf)

## 5. OneHotEncoding

Convert the categorical columns into numerical values using OneHotEncoding.

In [10]:
train_df_encoded = pd.get_dummies(train_df, columns=['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade'], dtype=int)
train_df_encoded.head()

,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,loan_paid_back,gender_Female,gender_Male,gender_Other,...,grade_subgrade_E1,grade_subgrade_E2,grade_subgrade_E3,grade_subgrade_E4,grade_subgrade_E5,grade_subgrade_F1,grade_subgrade_F2,grade_subgrade_F3,grade_subgrade_F4,grade_subgrade_F5
0,0,29367.99,0.084,736,2528.42,13.67,1.0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,22108.02,0.166,636,4593.10,12.92,0.0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,2,49566.20,0.097,694,17005.15,9.76,1.0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,3,46858.25,0.065,533,4682.48,16.10,1.0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
4,4,25496.70,0.053,665,12184.43,10.21,1.0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


Split the encoded dataset into 80% training data and 20% testing data.

In [11]:
X = train_df_encoded.drop(columns=['id', 'loan_paid_back'])
y = train_df_encoded['loan_paid_back']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=9)

### OneHotEncoding on Logistic Regression

In [12]:
my_logreg = LogisticRegression(max_iter=1000)
my_logreg.fit(X_train, y_train)

probability = my_logreg.predict_proba(X_test)[:, 1]
train_df_encoded = pd.get_dummies(train_df, columns=['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade'], dtype=int)
auc = roc_auc_score(y_test, probability)
print("Encoded Logistic Regression AUC:", auc)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Encoded Logistic Regression AUC: 0.8936709746408715


### OneHotEncoding on KNN

In [13]:
scaler = StandardScaler()
X_train_normalized = scaler.fit_transform(X_train)
X_test_normalized = scaler.transform(X_test)

k_values = [3, 5, 11, 15, 25, 30, 35, 45, 55, 75, 95, 125, 155, 200, 255]
auc_scores = []

for k in k_values:
    my_knn = KNeighborsClassifier(n_neighbors=k)
    my_knn.fit(X_train_normalized, y_train)
    probability = my_knn.predict_proba(X_test_normalized)[:, 1]
    auc = roc_auc_score(y_test, probability)
    auc_scores.append(auc)
    
for k, score in zip(k_values, auc_scores):
    print(f"K={k}: AUC {score:.4f}")

K=3: AUC 0.8252
K=5: AUC 0.8468
K=11: AUC 0.8697
K=15: AUC 0.8762
K=25: AUC 0.8845
K=30: AUC 0.8867
K=35: AUC 0.8881
K=45: AUC 0.8903
K=55: AUC 0.8912
K=75: AUC 0.8921
K=95: AUC 0.8920
K=125: AUC 0.8918
K=155: AUC 0.8918
K=200: AUC 0.8916
K=255: AUC 0.8910


### OneHotEncoding on Decision Tree

In [20]:
dt = DecisionTreeClassifier(random_state=5)
dt.fit(X_train, y_train)

probability_dt = dt.predict_proba(X_test)[:, 1]
auc_dt = roc_auc_score(y_test, probability_dt)
print("Decision Tree AUC:", auc_dt)

Decision Tree AUC: 0.7740693730322125


OneHotEncoding on Random Forest

In [ ]:
tree_counts = [100, 200, 300, 500]
rf_auc_scores = []

for n in tree_counts:
    my_rf = RandomForestClassifier(n_estimators=n, class_weight='balanced', random_state=5)
    
    my_rf.fit(X_train, y_train)
    
    probability = my_rf.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, probability)
    rf_auc_scores.append(auc)

for n, score in zip(tree_counts, rf_auc_scores):
    print(f"n_estimators={n}: AUC {score:.4f}")

## Final Results

In [ ]:
print("Final Model AUC Scores:")
print("Baseline Logistic Regression AUC: 0.7735")
print("Encoded Logistic Regression AUC:", auc_enc)
print("KNN AUC:", max(auc_scores))
print("Decision Tree AUC:", auc_dt)
print("Random Forest AUC:", auc_rf)